# Vector Embeddings & Vector Databases (running on LM Studio, 100% local)

**Goal of this notebook:** understand, from first principles, how an LLM app turns text into numbers it can compare — and why we need a special kind of database to store and search those numbers.

We'll cover:
1. What is a vector embedding?
2. Similar meaning = similar vectors (cosine similarity)
3. Why a plain list doesn't scale — the need for a vector database
4. Chroma — a simple, beginner-friendly vector database
5. Putting it together: a mini RAG pipeline

> We use **Chroma** (instead of FAISS) because its API (`add`, `query`) reads almost like plain English, and it stores your original text + metadata alongside the vectors automatically. FAISS is faster for huge production indexes, but it only stores raw vectors — you'd have to manage the text mapping yourself, which adds noise while you're still learning the *concept*.

> ⚠️ **This notebook runs entirely against your local LM Studio server — no API key, no cloud, no cost.** Because it talks to `localhost`, it must be run **locally** (e.g. in VS Code), not in Google Colab — Colab's servers can't reach your machine.

## 🖥️ Step 0 — Set up LM Studio

1. Install [LM Studio](https://lmstudio.ai/) and open it.
2. In the **search/discover** tab, download:
   - A small **chat** model, e.g. `LFM2.5 350M` or any Instruct model you like.
   - An **embedding** model, e.g. `text-embedding-nomic-embed-text-v1.5`.
3. Go to the **Developer** tab (the `</>` icon) and click **Start Server**. By default it runs at `http://localhost:1234`.
4. Run the cell below — it lists every model LM Studio currently knows about, so you can copy the exact model id strings into `CHAT_MODEL` and `EMBED_MODEL` further down.

In [1]:
%pip install -q openai chromadb numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from openai import OpenAI

LM_STUDIO_BASE_URL = "http://localhost:1234/v1"

# LM Studio doesn't check the API key, but the OpenAI client requires a non-empty string.
client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key="lm-studio")

print("Models available in LM Studio:")
for m in client.models.list().data:
    print(" -", m.id)

Models available in LM Studio:
 - lfm2.5-350m
 - text-embedding-nomic-embed-text-v1.5
 - text-embedding-multilingual-e5-large-instruct
 - text-embedding-multilingual-e5-small
 - text-embedding-all-minilm-l6-v2
 - text-embedding-all-minilm-l6-v2-embedding
 - text-embedding-bge-small-en-v1.5
 - lmstudio-community/gemma-4-e2b-it
 - gemma-4-e2b-it-qat
 - qwen3.5-4b
 - unsloth/gemma-4-e2b-it
 - gemma-3-270m-it-qat
 - google/gemma-4-e4b


👆 Copy the exact ids you downloaded into the two variables below (they must match what LM Studio printed above).

In [3]:
CHAT_MODEL = "lfm2.5-350m"                                  # any chat/instruct model you loaded
EMBED_MODEL = "text-embedding-nomic-embed-text-v1.5"          # any embedding model you loaded

## Part 1 — What is a vector embedding?

An **embedding** is just a list of numbers (a *vector*) that represents the *meaning* of a piece of text. A sentence about "dogs" and a sentence about "puppies" end up with vectors that point in roughly the same direction, even though they don't share many words.

Let's embed a single sentence — locally, via LM Studio — and look at what comes back.

In [4]:
response = client.embeddings.create(
    model=EMBED_MODEL,
    input="The cat sat on the mat.",
)

vector = response.data[0].embedding

print("How many numbers in the vector?", len(vector))
print("First 10 numbers:", vector[:10])

How many numbers in the vector? 768
First 10 numbers: [0.04953226447105408, 0.05858588591217995, -0.1163325086236, -0.030905580148100853, 0.043080754578113556, 0.05341465771198273, 0.012732407078146935, 0.04660245403647423, -0.040875744074583054, -0.04036561772227287]


Notice: a few hundred numbers for one short sentence (the exact count depends on which embedding model you loaded), and none of them mean anything on their own — `-0.0123` isn't "cat" and `0.045` isn't "mat". The *meaning* only shows up when you compare this vector to another one.

That comparison is the whole point of embeddings, so let's do it next.

## Part 2 — Similar meaning = similar vectors

To compare two vectors we use **cosine similarity**: it measures the angle between them, ignoring their length.

- `1.0` → pointing in the exact same direction (identical meaning)
- `0.0` → unrelated (perpendicular)
- `-1.0` → opposite meaning

Let's embed a few sentences — two about the same topic, one completely different — and compare them.

In [5]:
import numpy as np

def embed(text):
    result = client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(result.data[0].embedding)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [6]:
sentences = {
    "a": "The dog ran across the park.",
    "b": "A puppy sprinted through the garden.",
    "c": "The stock market fell sharply today.",
}

vectors = {key: embed(text) for key, text in sentences.items()}

print("a vs b (dog vs puppy)      :", round(float(cosine_similarity(vectors["a"], vectors["b"])), 4))
print("a vs c (dog vs stock market):", round(float(cosine_similarity(vectors["a"], vectors["c"])), 4))

a vs b (dog vs puppy)      : 0.7354
a vs c (dog vs stock market): 0.3909


You should see the **dog/puppy** pair score noticeably higher than the **dog/stock market** pair — even though "dog" and "puppy" don't share any letters in common. The model has captured *meaning*, not just keywords.

This is the core trick behind semantic search: turn text into vectors, then find the vectors that are closest together.

## Part 3 — Why do we need a *vector database*?

Comparing 2 vectors with `cosine_similarity` is easy. But real apps have thousands or millions of documents. If you had to:

- embed every document once,
- keep all those vectors in memory,
- and loop over *all of them* every time a user asks a question...

...that's slow, and it doesn't scale. A **vector database** solves this by:

1. **Storing** vectors alongside their original text + metadata (so you don't have to manage that mapping yourself).
2. **Indexing** vectors so "find the closest ones" is fast, even with millions of entries.
3. **Persisting** everything to disk so you don't re-embed on every run.

Let's use **Chroma**, a lightweight vector database, to see this in action — still fully local, still powered by LM Studio for the embeddings.

## Part 4 — Chroma: a simple vector database

Chroma's API has three ideas to learn:

- `client.create_collection(...)` — like creating a table
- `collection.add(...)` — insert documents (Chroma embeds them for you)
- `collection.query(...)` — ask "which documents are closest to this text?"

We'll point Chroma's `OpenAIEmbeddingFunction` at our LM Studio server (via `api_base`) instead of OpenAI's cloud, so the numbers stay consistent with what we just learned — and everything keeps running for free, on your machine.

In [7]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

lmstudio_ef = OpenAIEmbeddingFunction(
    api_key="lm-studio",
    api_base=LM_STUDIO_BASE_URL,
    model_name=EMBED_MODEL,
)

chroma_client = chromadb.Client()  # in-memory for this demo; use PersistentClient(path=...) to save to disk

collection = chroma_client.get_or_create_collection(
    name="bootcamp_demo",
    embedding_function=lmstudio_ef,
)

In [8]:
documents = [
    "The Eiffel Tower is located in Paris, France.",
    "Mount Everest is the tallest mountain above sea level.",
    "Python is a popular programming language for AI and data science.",
    "The Great Wall of China stretches over 13,000 miles.",
    "JavaScript is commonly used to build interactive websites.",
]

collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))],
)

print("Documents stored:", collection.count())

Documents stored: 5


In [9]:
results = collection.query(
    query_texts=["Which language should I learn for machine learning?"],
    n_results=2,
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"distance={distance:.4f}  ->  {doc}")

distance=0.3273  ->  Python is a popular programming language for AI and data science.
distance=0.4902  ->  JavaScript is commonly used to build interactive websites.


Chroma embedded our query (via LM Studio), compared it against every stored vector, and returned the **closest matches** — the documents about programming languages, not the ones about mountains or landmarks. Under the hood it did the exact same cosine-similarity idea from Part 2, just indexed and automated for us.

## Part 5 — Putting it together: a mini RAG pipeline

**RAG (Retrieval-Augmented Generation)** = retrieve relevant chunks from a vector database, then hand them to an LLM as context so it can answer using *your* data instead of just what it memorized during training.

The flow:

```
question -> embed -> search vector DB -> top-k chunks -> stuff into prompt -> local LLM answer
```

Both the retrieval *and* the generation step below run entirely on your machine through LM Studio.

In [10]:
def retrieve(query: str, k: int = 2) -> list[str]:
    results = collection.query(query_texts=[query], n_results=k)
    return results["documents"][0]

def ask(query: str) -> str:
    context = "\n".join(retrieve(query))
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer using only the context above."
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
    )
    return response.choices[0].message.content

print(ask("What is the tallest mountain?"))

Mount Everest


## Recap

- An **embedding** turns text into a vector of numbers that captures meaning.
- **Cosine similarity** compares two vectors to see how related they are.
- A **vector database** (like Chroma) stores many vectors + their original text, and makes "find the closest ones" fast at scale — that's what a plain Python list can't do well.
- **RAG** = retrieve the closest chunks from a vector database, then let the LLM answer using that context.
- **LM Studio** gave us both an embedding model and a chat model running locally, through the same OpenAI-compatible API — so the exact same code (`client.embeddings.create`, `client.chat.completions.create`) works whether you point `base_url` at OpenAI's cloud or at your own machine.

Next: try swapping in your own documents in Part 4, loading a bigger/different model in LM Studio and changing `CHAT_MODEL`/`EMBED_MODEL`, or explore `PersistentClient` so your Chroma collection survives across notebook restarts.